## Import das bibliotecas padrões:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import make_scorer, recall_score
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

## Import do dataset:

In [ ]:
df = pd.read_csv('../dados/processado/streaming_retencao_limpo.csv')

## Seleção de features:

Com base nas conclusões tomadas durante o processo de análise exploratória de dados junto com a seguinte análise do $\chi^2$ e p-value, as features selecionadas para o aprendizado de máquina são: 

`'dias_desde_ultimo_acesso' (0.33 corr),`

`'series_concluidas' (0.027 corr),`

`'series_abandonadas' (0.1 corr),`

`'tickets_suporte_abertos' (0.137 corr),`

`'genero_favorito' (p-value = 0.0213).`


### Cálculo de qui-quadrado e p-value:

In [ ]:
from scipy.stats import chi2_contingency

In [ ]:
churn_pagamento = pd.crosstab(df['pagamento'], df['cancelou'])

chi2, p_value, dof, frequencias_esperadas = chi2_contingency(churn_pagamento)

print(f"Estatística Qui-Quadrado (chi2): {chi2:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Graus de liberdade (dof): {dof}")

In [ ]:
churn_dispositivo = pd.crosstab(df['dispositivo_mais_acesso'], df['cancelou'])

chi2, p_value, dof, frequencias_esperadas = chi2_contingency(churn_dispositivo)

print(f"Estatística Qui-Quadrado (chi2): {chi2:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Graus de liberdade (dof): {dof}")

In [ ]:
churn_genero = pd.crosstab(df['genero_favorito'], df['cancelou'])

chi2, p_value, dof, frequencias_esperadas = chi2_contingency(churn_genero)

print(f"Estatística Qui-Quadrado (chi2): {chi2:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Graus de liberdade (dof): {dof}")

### Remoção das variáveis que não foram escolhidas:

In [ ]:
df = df.drop(columns = ['data_de_assinatura', 'idade_do_usuario', 'horas_consumidas', 'plano', 'pagamento', 'dispositivo_mais_acesso'])

## Encoding da variável `genero_favorito`:

Como não há uma hierarquia entre os valores desta variável, classiificamos-a como uma variável qualitativa nominal, sendo assim favorável para o uso da técnica de one-hot encoding.

In [ ]:
df = pd.get_dummies(df, columns=['genero_favorito'], drop_first=False, dtype=int)

## Padronização das Variáveis Numéricas

In [ ]:
scaler = StandardScaler()
colunas_numericas = ['dias_desde_ultimo_acesso', 'series_concluidas', 'series_abandonadas', 'tickets_suporte_abertos']
df[colunas_numericas] = scaler.fit_transform(df[colunas_numericas])


A fim de padronizar os dados, foi aplicado o StandardScaler apenas às colunas numéricas `dias_desde_ultimo_acesso`, `series_concluidas`, `series_abandonadas` e `tickets_suporte_abertos`.  As colunas categóricas foram preservadas na forma de hot encoding. 

## Divisão do Dataset

In [ ]:
X = df.drop(columns=["cancelou"])
y = df["cancelou"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
#stratify foi usado para manter a distribuição da variável cancelou entre treino e teste

### Verificação de distribuição da variável alvo:

In [ ]:
print("Proporção no conjunto de treino:")
print(y_train.value_counts(normalize=True))

print("\nProporção no conjunto de teste:")
print(y_test.value_counts(normalize=True))

A distribuição da variável alvo foi preservada nos conjuntos de treino e teste graças ao uso do parâmetro `stratify`.

# Treinamento do Modelo

In [ ]:
modelo_logistico = LogisticRegression(random_state=42)
modelo_logistico.fit(X_train, y_train)
y_pred = modelo_logistico.predict(X_test)

Por se tratar de um problema de classificação, foi cogitado usar modelos como Árvores de Decisão, Regressão Logística ou Random Forest. Diante disso, optou-se pelo modelo que mais se beneficiaria do StandardScaler, que é a Regressão Logística. Deste modo, podemos contar com um modelo que oferece alta interpretabilidade e resultados satisfatórios para o problema em questão.

## Matriz de Confusão

Com a análise da matriz de confusão fica claro que o modelo peca na predição dos positivos, tendo uma excelente capacidade de reconhecer os negativos e raramente ocorreno falsos positivos. Isso sugere um típico caso de overfitting.

In [ ]:
mc = confusion_matrix(y_test, y_pred)
vizualizacao_mc = ConfusionMatrixDisplay(confusion_matrix=mc, display_labels=modelo_logistico.classes_)
vizualizacao_mc.plot(cmap=plt.cm.Blues)
plt.title("Matriz de Confusão - Dados de Teste")
plt.show()

# Avaliação e Métricas de Classificação

Nesta etapa, vamos avaliar o desempenho do modelo utilizando métricas de classificação. Além da acurácia, serão analisadas métricas como precisão, revocação e F1-Score, que permitem entender melhor como o modelo se comporta em cada classe.

## Relatório de Classificação

O `classification_report` apresenta as principais métricas para cada classe prevista pelo modelo. Essas métricas serão utilizadas para analisar o desempenho da Regressão Logística na identificação dos clientes que cancelaram e dos que permaneceram ativos.

In [ ]:
relatorio = classification_report(
    y_test,
    y_pred,
    labels=["Não", "Sim"],
    target_names=["Não cancelou", "Cancelou"],
    output_dict=True,
    zero_division=0
)

tabela_relatorio = pd.DataFrame(relatorio).T

display(
    tabela_relatorio[
        ["precision", "recall", "f1-score", "support"]
    ].round(3)
)

### Interpretação dos Resultados

O modelo apresentou um desempenho muito bom para identificar os clientes que **não cancelaram**. Essa classe obteve precisão de **97,3%**, recall de **99,8%** e F1-Score de **98,5%**, indicando que praticamente todos os clientes ativos foram classificados corretamente.

Já para a classe **"Cancelou"**, o desempenho foi inferior. Embora a precisão tenha sido de **75%**, o recall ficou em apenas **18,8%**. Isso significa que, quando o modelo prevê um cancelamento, ele costuma acertar, mas identifica apenas uma pequena parte dos clientes que realmente cancelaram.

Esse comportamento provavelmente está relacionado ao desbalanceamento da base de dados, que possui muito mais exemplos da classe **"Não cancelou"** do que da classe **"Cancelou"**. Dessa forma, o modelo tende a aprender melhor os padrões da classe majoritária.

Para a SimonFlix, isso indica que o modelo ainda apresenta limitações para antecipar o churn. Apesar de gerar previsões de cancelamento relativamente confiáveis, muitos clientes que realmente cancelariam não são identificados, o que reduz a eficácia de possíveis ações de retenção.

## Acurácia do Modelo

Além das métricas apresentadas anteriormente, também foi calculada a acurácia do modelo. Essa métrica representa a porcentagem total de previsões corretas realizadas no conjunto de teste.

In [ ]:
acuracia = accuracy_score(y_test, y_pred)

print(f"Acurácia do modelo: {acuracia:.2%}")

### Interpretação da Acurácia

A Regressão Logística apresentou uma acurácia de **97,11%**, indicando que a maior parte das previsões realizadas no conjunto de teste foi classificada corretamente.

Entretanto, essa métrica deve ser analisada junto com o *classification report*. Apesar da alta acurácia, o modelo apresentou um recall de apenas **18,8%** para a classe **"Cancelou"**, mostrando que a maioria dos clientes que realmente cancelaram não foi identificada.

Esse resultado provavelmente está relacionado ao desbalanceamento da base de dados, que possui muito mais clientes da classe **"Não cancelou"**. Como consequência, o modelo tende a aprender melhor os padrões dessa classe, elevando a acurácia geral, mas reduzindo sua capacidade de identificar os clientes em risco de churn.

## Conclusão

A avaliação mostrou que a Regressão Logística apresentou um bom desempenho para identificar os clientes que permaneceram ativos, mas encontrou dificuldades para reconhecer aqueles que realmente cancelaram o serviço.

Embora a precisão da classe **"Cancelou"** tenha sido de **75%**, indicando que as previsões de churn costumam ser confiáveis, o recall de apenas **18,8%** demonstra que grande parte dos cancelamentos reais não foi identificada. Para a SimonFlix, esse é um ponto crítico, pois reduz a capacidade do modelo de antecipar clientes em risco e permitir ações de retenção.

Dessa forma, a Regressão Logística serviu como um bom ponto de partida para o problema, mas os resultados indicam que ainda há espaço para melhorias. Uma alternativa interessante para as próximas etapas seria testar algoritmos baseados em árvores de decisão, como o **Random Forest**, que costuma apresentar um desempenho superior em problemas de classificação com relações mais complexas entre as variáveis. Além disso, também podem ser exploradas técnicas de balanceamento da base de dados para aumentar a capacidade do modelo de identificar clientes que realmente irão cancelar o serviço.